# Exercise 2 — Diffusion from VACF and MSD: Choosing the Right Analysis Window

**MARVEL–ICTP School 2026**

You will run NVE production trajectories for SPC/Fw and TIP3P, then extract the self-diffusion coefficient $D$ of water by two independent methods:

1. **Green-Kubo (VACF)** — $D$ from the time integral of the velocity autocorrelation function
2. **Einstein (MSD)** — $D$ from the slope of the mean-squared displacement in the diffusive regime

The central skill in this exercise is **choosing the analysis window** for each method. A wrong window gives a wrong $D$ even from a perfect trajectory. You will also compare the two estimates and the two models against the experimental value.

## 1. Self-Diffusion from the VACF: Green-Kubo

The **velocity autocorrelation function** (VACF) of the molecular centre-of-mass measures how long a molecule remembers its direction of motion:

$$C(t) = \frac{1}{N}\sum_{i=1}^{N} \langle \mathbf{v}_i(0) \cdot \mathbf{v}_i(t) \rangle$$

In a liquid, $C(t)$ decays from $C(0) = \langle v^2 \rangle$ to zero on a timescale of ~1 ps. In dense liquids such as water, the decay is not monotone: $C(t)$ goes slightly negative (the **cage effect** — a molecule bounces back from its neighbours before diffusing freely), then returns to zero.

The **Green-Kubo relation** connects the VACF to the self-diffusion coefficient:

$$D = \frac{1}{3} \int_0^\infty C(t)\, dt$$

In practice the integral is cut at a finite time $t_\text{max}$. The **running integral**

$$D(t) = \frac{1}{3} \int_0^t C(t')\, dt'$$

should plateau once $C(t')$ has decayed to noise. The plateau value is $D$.

### How to choose $t_\text{max}$

- **Too short**: $C(t)$ has not yet decayed — the integral is still rising — $D$ is underestimated.
- **Too long**: $C(t)$ is just noise around zero — accumulated random contributions make $D(t)$ drift — estimate becomes noisy.
- **Correct**: $D(t)$ has reached a clear plateau. Inspect the plot and choose $t_\text{max}$ at the beginning of the plateau.


## 2. Self-Diffusion from the MSD: Einstein Relation

The **mean-squared displacement** (MSD) tracks how far molecules move on average:

$$\text{MSD}(t) = \frac{1}{N}\sum_{i=1}^{N} \langle |\mathbf{r}_i(t) - \mathbf{r}_i(0)|^2 \rangle$$

At short times the motion is ballistic ($\text{MSD} \propto t^2$); at long times it is diffusive ($\text{MSD} \propto t$). The **Einstein relation** gives $D$ from the diffusive slope:

$$D = \frac{1}{6} \lim_{t \to \infty} \frac{\mathrm{d}\,\text{MSD}}{\mathrm{d}t}$$

### How to choose the fit window $[t_\text{start}, t_\text{end}]$

- **$t_\text{start}$ too small**: fit includes the ballistic regime (slope 2 on log-log) → overestimates $D$.
- **$t_\text{end}$ too large**: at long lag times few time-origin pairs are available → MSD is poorly averaged and noisy → $D$ is unreliable.
- **Correct**: fit only the linear diffusive portion. A log-log plot makes the crossover from ballistic to diffusive visually obvious. Rule of thumb: fit between $\sim$1 ps and $\sim$20–30% of the total trajectory length.


## 3. Instructions

### Step 1 — Run the NVE production for both models

From the `exercise_2/` directory:

```bash
cd simulations/spcfw_prod
lmp -in ../../in.spcfw.prod.lammps

cd simulations/tip3p_prod
lmp -in ../../in.tip3p.prod.lammps
```

If you want, you can also launch both runs with:
```bash
bash run_exercise_2_sims.sh
```

Each run produces `traj.lammpstrj` (trajectory with positions and velocities) and `log.lammps`. The default is 100 ps; use `-var nsteps 400000` for 200 ps if time allows — longer trajectories give better statistics for both VACF and MSD.

### Step 2 — Pre-compute VACF and MSD (run the cell below once)

The **Pre-computation** cell reads the trajectories and saves VACF and MSD as CSV files. This takes a few minutes and needs to be run only once per trajectory.

### Step 3 — Set your analysis windows

Fill in `VACF_TMAX_FS` and the MSD fit range in the **Student configuration** cell, then re-run the analysis and plot cells. These cells load from the CSVs and are fast to re-run.

In [18]:
from __future__ import annotations
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.rcParams.update({
    "figure.dpi": 130,
    "savefig.dpi": 200,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

ROOT   = Path(".").resolve()
SIM    = ROOT / "simulations"
sys.path.insert(0, str(ROOT / "analysis_scripts"))

# Physical constants for unit conversion (from msd_vacf.py)
T_AU_FS              = 2.4188843265857e-2   # fs per a.u. of time
BOHR_M               = 5.29177210903e-11    # m per bohr
T_AU_S               = 2.4188843265857e-17  # s per a.u. of time
V_AFS_TO_AU          = T_AU_FS / (BOHR_M * 1e10)  # (Ang/fs) -> a.u.
AU_DIFF_TO_1E5_CM2_S = (BOHR_M**2 / T_AU_S) * 1e7 * 1e2  # a.u. -> 1e-5 cm^2/s

D_EXP = 2.30        # experimental D of water at 298 K [1e-5 cm^2/s]
D_TIP3P_REF = 4.03  # TIP3P(Ewald) reference from Price and Brooks [1e-5 cm^2/s]
D_MODEL_REF = {"spcfw": D_EXP, "tip3p": D_TIP3P_REF}

MODELS = {
    "spcfw": {"label": "SPC/Fw",          "color": "#1f77b4"},
    "tip3p": {"label": "TIP3P (rigid)",   "color": "#d62728"},
}

print(f"Root: {ROOT}")

Root: /Users/federica/Documents/Teaching activity/Marvel_college/exercises/exercise_2


## 4. Pre-computation: VACF and MSD from Trajectories

Run the cell below **once** after each LAMMPS production run. It reads the full trajectory, computes the VACF (via FFT) and the MSD (multi-origin), and saves both to CSV files in the simulation directories. This cell takes a few minutes; subsequent analysis cells load from the CSVs and are fast.

The trajectory dump interval is 100 steps × 0.5 fs = **50 fs per frame**.

In [19]:
import msd_vacf as mv

DUMP_DT_FS  = 50.0   # physical time between trajectory frames [fs]
TYPE_MAP    = {1: "O", 2: "H"}
SPECIES     = ["O", "H"]

for model in ("spcfw", "tip3p"):
    traj = SIM / f"{model}_prod" / "traj.lammpstrj"
    out  = SIM / f"{model}_prod"
    if not traj.exists():
        print(f"[{model}] trajectory not found — skipping ({traj})")
        continue
    print(f"[{model}] computing VACF and MSD from {traj.name} ...")
    mv.run_unified(
        input_path         = str(traj),
        species            = SPECIES,
        mode               = "from-vel",
        input_format       = "lammps",
        dt_fs              = DUMP_DT_FS,
        output_dir         = str(out),
        lammps_type_map    = TYPE_MAP,
        compute_water_self = True,
        compute_msd        = True,
        compute_vacf       = True,
        water_self_target  = "com",
    )
    print(f"[{model}] done.")

[spcfw] trajectory not found — skipping (/Users/federica/Documents/Teaching activity/Marvel_college/exercises/exercise_2/simulations/spcfw_prod/traj.lammpstrj)
[tip3p] trajectory not found — skipping (/Users/federica/Documents/Teaching activity/Marvel_college/exercises/exercise_2/simulations/tip3p_prod/traj.lammpstrj)


## 5. Student Configuration

Set the VACF integration limit and the MSD fit window here, then re-run the analysis cells below.

In [17]:
# ============================================================
# STUDENT CONFIGURATION — fill in after inspecting the plots
# ============================================================

# VACF integration window [fs]
# Set to the time where D(t) (running integral) clearly plateaus.
# Start with None to see the full curve, then narrow it down.
VACF_TMAX_FS = {
    "spcfw": None,   # e.g. 3000
    "tip3p": None,   # e.g. 2000
}

# MSD linear fit window [fs]
# Choose the range where MSD is already diffusive (slope 1 on log-log)
# but still well-averaged (not too close to the trajectory length).
MSD_FIT_START_FS = {
    "spcfw": None,   # e.g. 2000
    "tip3p": None,   # e.g. 2000
}
MSD_FIT_END_FS = {
    "spcfw": None,   # e.g. 30000
    "tip3p": None,   # e.g. 30000
}

## 6. Analysis: VACF

The left panel shows the normalised VACF $C(t)/C(0)$. Look for:
- the initial decay (velocity decorrelation)
- the negative dip (cage effect — molecule bounces back)
- the return to zero (end of memory)

The right panel shows the **running integral** $D(t) = \frac{1}{3}\int_0^t C(t')\,dt'$. Choose `VACF_TMAX_FS` where $D(t)$ reaches a clear plateau. If the plateau is not visible, the trajectory is too short.

In [20]:
vacf_data: dict[str, dict] = {}

for model in ("spcfw", "tip3p"):
    csv = SIM / f"{model}_prod" / "vacf_H2O_COM.csv"
    if not csv.exists():
        print(f"[{model}] VACF CSV not found — run the pre-computation cell first.")
        continue
    df = pd.read_csv(csv)
    t  = df["time_fs"].to_numpy()
    c  = df["vacf"].to_numpy()          # a.u. units, already per-component (= C_vector/3)
    cn = df["vacf_normalized"].to_numpy()
    dt = float(t[1] - t[0]) if len(t) > 1 else DUMP_DT_FS

    # Running integral D(t) in units of 1e-5 cm^2/s
    # The CSV stores the per-component VACF (C_vector/3), so:
    #   D = integral of C_per_component  (no extra 1/3 factor)
    dt_au = dt / T_AU_FS
    d_t   = mv.cumulative_integrate_trapz(c, dt_au) * AU_DIFF_TO_1E5_CM2_S

    # Truncate at VACF_TMAX_FS if set
    tmax = VACF_TMAX_FS[model]
    if tmax is not None:
        mask = t <= tmax
        t, c, cn, d_t = t[mask], c[mask], cn[mask], d_t[mask]

    vacf_data[model] = {"t": t, "vacf": c, "vacf_norm": cn, "D_t": d_t, "dt": dt}
    D_final = float(d_t[-1])
    print(f"[{model}] D_VACF = {D_final:.3f}  [1e-5 cm^2/s]  "
          f"(tmax = {t[-1]/1000:.1f} ps)")

[spcfw] VACF CSV not found — run the pre-computation cell first.
[tip3p] VACF CSV not found — run the pre-computation cell first.


In [21]:
if not vacf_data:
    print("No VACF data loaded.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    for model, meta in MODELS.items():
        if model not in vacf_data:
            continue
        d   = vacf_data[model]
        t_ps = d["t"] / 1000
        c   = meta["color"]
        lbl = meta["label"]

        axes[0].plot(t_ps, d["vacf_norm"], color=c, lw=1.5, label=lbl)
        axes[1].plot(t_ps, d["D_t"],       color=c, lw=2.0, label=lbl)

    axes[0].axhline(0, color="k", lw=0.8, ls="--")
    axes[0].set_xlabel("Time [ps]")
    axes[0].set_ylabel("$C(t) / C(0)$")
    axes[0].set_title("Normalised VACF")
    axes[0].legend(frameon=False)

    axes[1].axhline(D_EXP, color="k", lw=1.0, ls=":",
                    label=f"Experiment: {D_EXP} × 10⁻⁵ cm²/s")
    axes[1].axhline(D_TIP3P_REF, color=MODELS["tip3p"]["color"], lw=1.0, ls=":",
                    label=f"TIP3P(Ewald) ref.: {D_TIP3P_REF} × 10⁻⁵ cm²/s")
    axes[1].set_xlabel("Time [ps]")
    axes[1].set_ylabel(r"$D(t)$ [$10^{-5}$ cm$^2$/s]")
    axes[1].set_title("Running integral — identify the plateau")
    axes[1].legend(frameon=False)

    fig.tight_layout()
    plt.show()

No VACF data loaded.


**Discussion — VACF window.**
What happens to $D_\text{VACF}$ if you cut the integral too early (before the plateau) or too late (deep into the noise)? Quantify the effect by trying two additional values of `VACF_TMAX_FS` and comparing the results.

## 7. Analysis: MSD

The **log-log plot** (left) shows the crossover from ballistic ($\text{MSD} \propto t^2$, slope 2) to diffusive ($\text{MSD} \propto t$, slope 1). Use it to identify the onset of the diffusive regime and set `MSD_FIT_START_FS` just after the crossover.

The **linear plot** (right) shows the MSD with the fitted line. The slope gives $D = \frac{1}{6}\,\text{d(MSD)}/\text{dt}$.

In [22]:
msd_data: dict[str, dict] = {}

for model in ("spcfw", "tip3p"):
    csv = SIM / f"{model}_prod" / "msd_H2O_COM.csv"
    if not csv.exists():
        print(f"[{model}] MSD CSV not found — run the pre-computation cell first.")
        continue
    df  = pd.read_csv(csv)
    t   = df["time_fs"].to_numpy()
    msd = df["msd_A2"].to_numpy()    # Ang^2
    dt  = float(t[1] - t[0]) if len(t) > 1 else DUMP_DT_FS

    t0 = MSD_FIT_START_FS[model]
    t1 = MSD_FIT_END_FS[model]
    res = mv.diffusion_from_msd(msd, dt, dim=3,
                                fit_start_fs=t0, fit_end_fs=t1)
    D_msd = res.get("D_msd_1e-5_cm^2/s", float("nan"))
    fit_start = res.get("D_msd_fit_start_fs")
    fit_end   = res.get("D_msd_fit_end_fs")

    slope = float("nan")
    intercept = float("nan")
    r2 = float("nan")
    n_fit = 0
    if fit_start is not None and fit_end is not None:
        mask_fit = (t >= fit_start) & (t <= fit_end)
        n_fit = int(mask_fit.sum())
        if n_fit > 1:
            t_fit = t[mask_fit]
            msd_fit = msd[mask_fit]
            dt_centered = t_fit - t_fit.mean()
            slope = float(np.dot(dt_centered, msd_fit - msd_fit.mean()) / np.dot(dt_centered, dt_centered))
            intercept = float(msd_fit.mean() - slope * t_fit.mean())
            msd_pred = slope * t_fit + intercept
            ss_res = float(np.sum((msd_fit - msd_pred)**2))
            ss_tot = float(np.sum((msd_fit - msd_fit.mean())**2))
            r2 = 1.0 - ss_res / ss_tot if ss_tot > 0 else float("nan")

    msd_data[model] = {
        "t": t, "msd": msd, "dt": dt,
        "fit_start": fit_start,
        "fit_end":   fit_end,
        "fit_slope": slope,
        "fit_intercept": intercept,
        "fit_r2": r2,
        "fit_n": n_fit,
        "D_msd":     D_msd,
    }
    print(f"[{model}] D_MSD  = {D_msd:.3f}  [1e-5 cm^2/s]  "
          f"(fit: {fit_start/1000 if fit_start is not None else 0:.1f}–"
          f"{fit_end/1000 if fit_end is not None else 0:.1f} ps, "
          f"R^2={r2:.4f}, n={n_fit})")

[spcfw] MSD CSV not found — run the pre-computation cell first.
[tip3p] MSD CSV not found — run the pre-computation cell first.


In [23]:
if not msd_data:
    print("No MSD data loaded.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    # Reference slopes for log-log, anchored to the data so they sit in the right range.
    t_ref_ps = np.geomspace(0.05, 10.0, 200)
    t_ballistic_anchor_ps = 0.2
    t_diffusive_anchor_ps = 3.0
    msd_ballistic_anchor = []
    msd_diffusive_anchor = []
    for d in msd_data.values():
        t_ps_all = d["t"] / 1000
        msd_all = d["msd"]
        mask_pos = msd_all > 0
        if not np.any(mask_pos):
            continue
        i_ball = np.argmin(np.abs(t_ps_all[mask_pos] - t_ballistic_anchor_ps))
        i_diff = np.argmin(np.abs(t_ps_all[mask_pos] - t_diffusive_anchor_ps))
        msd_ballistic_anchor.append(msd_all[mask_pos][i_ball])
        msd_diffusive_anchor.append(msd_all[mask_pos][i_diff])
    if msd_ballistic_anchor and msd_diffusive_anchor:
        y_ball = float(np.median(msd_ballistic_anchor))
        y_diff = float(np.median(msd_diffusive_anchor))
        axes[0].plot(t_ref_ps, y_ball * (t_ref_ps / t_ballistic_anchor_ps)**2,
                     'k:', lw=1, label="slope 2 (ballistic)")
        axes[0].plot(t_ref_ps, y_diff * (t_ref_ps / t_diffusive_anchor_ps),
                     'k--', lw=1, label="slope 1 (diffusive)")

    for model, meta in MODELS.items():
        if model not in msd_data:
            continue
        d    = msd_data[model]
        t_ps = d["t"] / 1000
        c    = meta["color"]
        lbl  = meta["label"]
        D    = d["D_msd"]
        t0   = d["fit_start"]
        t1   = d["fit_end"]
        m    = d.get("fit_slope", float("nan"))
        q    = d.get("fit_intercept", float("nan"))
        r2   = d.get("fit_r2", float("nan"))

        # log-log
        mask_pos = d["msd"] > 0
        axes[0].loglog(t_ps[mask_pos], d["msd"][mask_pos], color=c, lw=1.2, label=lbl)

        # linear with fit
        axes[1].plot(t_ps, d["msd"], color=c, lw=1.2, alpha=0.7, label=lbl)
        if not np.isnan(D) and t0 is not None and t1 is not None and not np.isnan(m) and not np.isnan(q):
            mask_fit = (d["t"] >= t0) & (d["t"] <= t1)
            if mask_fit.sum() > 1:
                t_w = d["t"][mask_fit]
                msd_line = m * t_w + q
                fit_lbl = f"{lbl} fit: D={D:.3f}"
                if not np.isnan(r2):
                    fit_lbl += f", R^2={r2:.4f}"
                axes[1].plot(t_w/1000, msd_line, color=c, lw=2.5, ls="--",
                             label=fit_lbl)
                axes[1].axvspan(t0/1000, t1/1000, alpha=0.08, color=c)

    axes[0].set_xlabel("Time [ps]")
    axes[0].set_ylabel(r"MSD [Å$^2$]")
    axes[0].set_title("MSD log-log — identify diffusive regime")
    axes[0].legend(frameon=False, fontsize=8)

    axes[1].set_xlabel("Time [ps]")
    axes[1].set_ylabel(r"MSD [Å$^2$]")
    axes[1].set_title("MSD linear — fit window (shaded)")
    axes[1].legend(frameon=False, fontsize=8)

    fig.tight_layout()
    plt.show()

No MSD data loaded.


**Discussion — MSD window.**
Move `MSD_FIT_START_FS` into the ballistic regime (e.g. 100 fs). How much does $D_\text{MSD}$ change? What does this tell you about the importance of the fit window choice?

## 8. Comparison: $D_\text{VACF}$ vs $D_\text{MSD}$

In [24]:
rows = []
for model, meta in MODELS.items():
    D_vacf = float(vacf_data[model]["D_t"][-1]) if model in vacf_data else float("nan")
    D_msd  = msd_data[model]["D_msd"]           if model in msd_data  else float("nan")
    rows.append({
        "model":                  meta["label"],
        "D_VACF [1e-5 cm²/s]": round(D_vacf, 3),
        "D_MSD  [1e-5 cm²/s]": round(D_msd,  3),
        "D_exp  [1e-5 cm²/s]": D_EXP,
        "D_ref model [1e-5 cm²/s]": D_MODEL_REF[model],
        "err_VACF vs exp [%]": round(100 * (D_vacf - D_EXP) / D_EXP, 1) if not np.isnan(D_vacf) else "—",
        "err_MSD  vs exp [%]": round(100 * (D_msd  - D_EXP) / D_EXP, 1) if not np.isnan(D_msd)  else "—",
        "err_VACF vs model ref [%]": round(100 * (D_vacf - D_MODEL_REF[model]) / D_MODEL_REF[model], 1) if not np.isnan(D_vacf) else "—",
        "err_MSD  vs model ref [%]": round(100 * (D_msd  - D_MODEL_REF[model]) / D_MODEL_REF[model], 1) if not np.isnan(D_msd)  else "—",
    })

display(pd.DataFrame(rows).set_index("model"))

print("\nConsistency check: D_VACF and D_MSD should agree within statistical uncertainty.")
print("If they disagree by more than ~10%, one of the windows is likely wrong.")

,D_VACF [1e-5 cm²/s],D_MSD [1e-5 cm²/s],D_exp [1e-5 cm²/s],err_VACF vs exp [%],err_MSD vs exp [%]
model,,,,,
SPC/Fw,NaN,NaN,2.3,—,—
TIP3P (rigid),NaN,NaN,2.3,—,—



Consistency check: D_VACF and D_MSD should agree within statistical uncertainty.
If they disagree by more than ~10%, one of the windows is likely wrong.


**Discussion — Consistency and model comparison.**

3. Do $D_\text{VACF}$ and $D_\text{MSD}$ agree for the same model? If not, which window is more likely to be wrong, and why?

4. Compare your TIP3P result both to the experimental value $D = 2.30 \times 10^{-5}$ cm²/s and to the literature TIP3P(Ewald) reference $D = 4.03 \times 10^{-5}$ cm²/s. What does this tell you about model-specific expectations versus experiment?